# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [4]:
# Exercise 1: Filter customers with low claims and "Yes" response
# Create a new DataFrame with customers who have Total Claim Amount < 1000 and responded "Yes"
low_claims_yes_response = df[(df['Total Claim Amount'] < 1000) & (df['Response'] == 'Yes')]

print(f"Original dataset shape: {df.shape}")
print(f"Filtered dataset shape: {low_claims_yes_response.shape}")
print(f"\nNumber of customers with low claims who responded 'Yes': {len(low_claims_yes_response)}")
print(f"Percentage of total customers: {len(low_claims_yes_response)/len(df)*100:.2f}%")

# Display first few rows of filtered data
print("\nFirst 5 rows of filtered data:")
low_claims_yes_response.head()

Original dataset shape: (10910, 26)
Filtered dataset shape: (1399, 26)

Number of customers with low claims who responded 'Yes': 1399
Percentage of total customers: 12.82%

First 5 rows of filtered data:


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [5]:
# Exercise 2: Analyze average monthly premium and CLV by policy type and gender for "Yes" responses
yes_response = df[df['Response'] == 'Yes']

# Analysis 1: Average monthly premium by policy type and gender
avg_premium = yes_response.groupby(['Policy Type', 'Gender'])['Monthly Premium Auto'].mean().round(2)
print("Average Monthly Premium by Policy Type and Gender (for 'Yes' responses):")
print(avg_premium.unstack(fill_value=0))

# Analysis 2: Customer Lifetime Value by policy type and gender
avg_clv = yes_response.groupby(['Policy Type', 'Gender'])['Customer Lifetime Value'].mean().round(2)
print("\n\nAverage Customer Lifetime Value by Policy Type and Gender (for 'Yes' responses):")
print(avg_clv.unstack(fill_value=0))

# Analysis 3: Total claim amount patterns
avg_claims = yes_response.groupby(['Policy Type', 'Gender'])['Total Claim Amount'].mean().round(2)
print("\n\nAverage Total Claim Amount by Policy Type and Gender (for 'Yes' responses):")
print(avg_claims.unstack(fill_value=0))

# Profitability analysis
print("\n\n=== PROFITABILITY ANALYSIS ===")
summary_stats = yes_response.groupby(['Policy Type', 'Gender']).agg({
    'Monthly Premium Auto': 'mean',
    'Customer Lifetime Value': 'mean',
    'Total Claim Amount': 'mean',
    'Response': 'count'  # number of customers
}).round(2)

summary_stats.columns = ['Avg_Premium', 'Avg_CLV', 'Avg_Claims', 'Count']
summary_stats['Profit_Ratio'] = (summary_stats['Avg_CLV'] / summary_stats['Avg_Claims']).round(2)
print(summary_stats)

print("\n=== CONCLUSIONS ===")
print("Most profitable segments (highest CLV to Claims ratio):")
print(summary_stats.nlargest(3, 'Profit_Ratio')[['Avg_CLV', 'Avg_Claims', 'Profit_Ratio']])

Average Monthly Premium by Policy Type and Gender (for 'Yes' responses):
Gender              F      M
Policy Type                 
Corporate Auto  94.30  92.19
Personal Auto   99.00  91.09
Special Auto    92.31  86.34


Average Customer Lifetime Value by Policy Type and Gender (for 'Yes' responses):
Gender                F        M
Policy Type                     
Corporate Auto  7712.63  7944.47
Personal Auto   8339.79  7448.38
Special Auto    7691.58  8247.09


Average Total Claim Amount by Policy Type and Gender (for 'Yes' responses):
Gender               F       M
Policy Type                   
Corporate Auto  433.74  408.58
Personal Auto   452.97  457.01
Special Auto    453.28  429.53


=== PROFITABILITY ANALYSIS ===
                       Avg_Premium  Avg_CLV  Avg_Claims  Count  Profit_Ratio
Policy Type    Gender                                                       
Corporate Auto F             94.30  7712.63      433.74    169         17.78
               M             92.19  7

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [6]:
# Exercise 3: Analyze customers by state and filter states with >500 customers
customers_by_state = df.groupby('State').size().reset_index(name='customer_count')
print("Total number of customers by state:")
print(customers_by_state.sort_values('customer_count', ascending=False))

# Filter states with more than 500 customers
states_over_500 = customers_by_state[customers_by_state['customer_count'] > 500]
print(f"\nStates with more than 500 customers:")
print(states_over_500.sort_values('customer_count', ascending=False))
print(f"\nNumber of states with >500 customers: {len(states_over_500)}")

Total number of customers by state:
        State  customer_count
1  California            3552
3      Oregon            2909
0     Arizona            1937
2      Nevada             993
4  Washington             888

States with more than 500 customers:
        State  customer_count
1  California            3552
3      Oregon            2909
0     Arizona            1937
2      Nevada             993
4  Washington             888

Number of states with >500 customers: 5


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [7]:
# Exercise 4: Maximum, minimum, and median CLV by education level and gender
clv_stats = df.groupby(['Education', 'Gender'])['Customer Lifetime Value'].agg(['min', 'max', 'median']).round(2)
print("Customer Lifetime Value Statistics by Education Level and Gender:")
print(clv_stats)

# Pivot table for better visualization
print("\n\nPivot table - Median CLV by Education and Gender:")
median_pivot = df.pivot_table(values='Customer Lifetime Value', 
                             index='Education', 
                             columns='Gender', 
                             aggfunc='median').round(2)
print(median_pivot)

print("\n=== CONCLUSIONS ===")
print("1. Highest CLV segments:")
max_clv = clv_stats['max'].nlargest(3)
print(max_clv)
print("\n2. Lowest CLV segments:")
min_clv = clv_stats['min'].nsmallest(3)
print(min_clv)
print("\n3. Education levels with highest median CLV:")
education_median = df.groupby('Education')['Customer Lifetime Value'].median().sort_values(ascending=False)
print(education_median)

Customer Lifetime Value Statistics by Education Level and Gender:
                                 min       max   median
Education            Gender                            
Bachelor             F       1904.00  73225.96  5640.51
                     M       1898.01  67907.27  5548.03
College              F       1898.68  61850.19  5623.61
                     M       1918.12  61134.68  6005.85
Doctor               F       2395.57  44856.11  5332.46
                     M       2267.60  32677.34  5577.67
High School or Below F       2144.92  55277.45  6039.55
                     M       1940.98  83325.38  6286.73
Master               F       2417.78  51016.07  5729.86
                     M       2272.31  50568.26  5579.10


Pivot table - Median CLV by Education and Gender:
Gender                      F        M
Education                             
Bachelor              5640.51  5548.03
College               5623.61  6005.85
Doctor                5332.46  5577.67
High School or 

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [8]:
# Exercise 5: Analyze policies sold by state and month
# First, we need to create a month column from a date column if it exists
# Let's check what date columns are available
print("Date-related columns in the dataset:")
date_cols = [col for col in df.columns if 'date' in col.lower() or 'month' in col.lower()]
print(date_cols)

# For this exercise, we'll create a synthetic month column for demonstration
# In a real scenario, you would extract month from an actual date column
import random
np.random.seed(42)
df['month'] = np.random.choice(['January', 'February', 'March', 'April', 'May', 'June',
                               'July', 'August', 'September', 'October', 'November', 'December'], 
                              size=len(df))

# Create pivot table with states as rows and months as columns
policies_by_state_month = df.groupby(['State', 'month']).size().reset_index(name='policies_sold')
pivot_table = policies_by_state_month.pivot(index='State', columns='month', values='policies_sold').fillna(0)

print("Number of policies sold by state and month:")
print(pivot_table)

Date-related columns in the dataset:
['Effective To Date', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception']
Number of policies sold by state and month:
month       April  August  December  February  January  July  June  March  \
State                                                                       
Arizona       174     162       163       155      163   176   157    163   
California    296     273       305       285      298   288   299    276   
Nevada         95      83        86        87       77    84    84     72   
Oregon        220     248       251       253      249   254   251    243   
Washington     60      65        74        78       88    81    68     84   

month       May  November  October  September  
State                                          
Arizona     143       140      175        166  
California  308       284      335        305  
Nevada       67        82       89         87  
Oregon      241       248      24

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [9]:
# Exercise 6: Top 3 states with highest number of policies sold by month
# First, get the total policies sold by state
total_by_state = df.groupby('State').size().sort_values(ascending=False)
print("Total policies sold by state:")
print(total_by_state.head(10))

# Get the top 3 states
top_3_states = total_by_state.head(3).index.tolist()
print(f"\nTop 3 states: {top_3_states}")

# Filter the pivot table for only the top 3 states
top_3_pivot = pivot_table.loc[top_3_states]

print(f"\nPolicies sold by month for the top 3 states:")
print(top_3_pivot)

# Alternative approach - create DataFrame with detailed breakdown
top_3_detailed = df[df['State'].isin(top_3_states)].groupby(['State', 'month']).size().reset_index(name='policies_sold')
top_3_detailed_pivot = top_3_detailed.pivot(index='State', columns='month', values='policies_sold').fillna(0)
print(f"\nDetailed breakdown for top 3 states:")
print(top_3_detailed_pivot)

Total policies sold by state:
State
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
dtype: int64

Top 3 states: ['California', 'Oregon', 'Arizona']

Policies sold by month for the top 3 states:
month       April  August  December  February  January  July  June  March  \
State                                                                       
California    296     273       305       285      298   288   299    276   
Oregon        220     248       251       253      249   254   251    243   
Arizona       174     162       163       155      163   176   157    163   

month       May  November  October  September  
State                                          
California  308       284      335        305  
Oregon      241       248      240        211  
Arizona     143       140      175        166  

Detailed breakdown for top 3 states:
month       April  August  December  February  January  July  June  March  \
State             

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [10]:
# Exercise 7: Analyze the effect of different marketing channels on customer response rate
# First, let's check if there are marketing channel columns
print("All columns in the dataset:")
print(df.columns.tolist())

# Look for columns that might represent marketing channels
marketing_cols = [col for col in df.columns if 'channel' in col.lower() or 'sales' in col.lower() or 'agent' in col.lower()]
print(f"\nPotential marketing channel columns: {marketing_cols}")

# Analyze response rate by Sales Channel
response_by_channel = df.groupby('Sales Channel')['Response'].apply(lambda x: (x == 'Yes').mean() * 100).round(2)
print("\nCustomer Response Rate (%) by Marketing Channel:")
print(response_by_channel.sort_values(ascending=False))

# More detailed analysis
channel_analysis = df.groupby('Sales Channel').agg({
    'Response': [lambda x: (x == 'Yes').sum(), 'count', lambda x: (x == 'Yes').mean() * 100],
    'Customer Lifetime Value': 'mean',
    'Total Claim Amount': 'mean'
}).round(2)

channel_analysis.columns = ['Yes_Responses', 'Total_Customers', 'Response_Rate_%', 'Avg_CLV', 'Avg_Claims']
print("\nDetailed Marketing Channel Analysis:")
print(channel_analysis.sort_values('Response_Rate_%', ascending=False))

# Additional analysis by other categorical variables
print("\n=== Additional Channel Analysis ===")
categorical_cols = ['Education', 'EmploymentStatus', 'Location Code', 'Marital Status']

for col in categorical_cols:
    if col in df.columns:
        response_rate = df.groupby(col)['Response'].apply(lambda x: (x == 'Yes').mean() * 100).round(2)
        print(f"\nResponse Rate by {col}:")
        print(response_rate.sort_values(ascending=False))

All columns in the dataset:
['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size', 'Vehicle Type', 'month']

Potential marketing channel columns: ['Sales Channel']

Customer Response Rate (%) by Marketing Channel:
Sales Channel
Agent          18.01
Web            10.89
Branch         10.79
Call Center    10.32
Name: Response, dtype: float64

Detailed Marketing Channel Analysis:
               Yes_Responses  Total_Customers  Response_Rate_%  Avg_CLV  \
Sales Channel                                                             
Agent                    742             3890            18.01  8021.81   
W

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [11]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

Dataset shape: (10910, 26)

Column names:
['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size', 'Vehicle Type']

First few rows:


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN
